# cuVS - Basic Uses

cuVS searches vector embeddings on the GPU. It does not know about molecules directly; in this notebook, SMILES strings become dense vectors and cuVS searches those vectors by cosine distance.

What's in here:
1. Build a vector matrix and run exact k-nearest-neighbor search
2. Join neighbor IDs back to molecule metadata
3. Convert between CuPy GPU arrays and NumPy CPU arrays
4. Use cuVS in an embedding workflow
5. Compare exact `brute_force` with approximate IVF-Flat

In [ ]:
import cupy as cp
import numpy as np
import pandas as pd
import torch
import smirk
from cuvs.neighbors import brute_force, ivf_flat
from transformers import AutoModel, AutoTokenizer

print(f"GPU: {cp.cuda.runtime.getDeviceProperties(0)['name'].decode()}")

molecules = pd.DataFrame({
    "name": ["ethanol", "propanol", "isopropanol", "benzene", "phenol", "toluene", "aspirin", "ibuprofen"],
    "smiles": ["CCO", "CCCO", "CC(C)O", "c1ccccc1", "c1ccccc1O", "Cc1ccccc1", "CC(=O)Oc1ccccc1C(=O)O", "CC(C)Cc1ccc(C(C)C(=O)O)cc1"],
    "family": ["alcohol", "alcohol", "alcohol", "aromatic", "aromatic", "aromatic", "drug-like", "drug-like"],
})

MODEL_NAME = "mist-models/mist-28M-ti624ev1"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModel.from_pretrained(MODEL_NAME, trust_remote_code=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device).eval()

def embed_smiles(smiles_list):
    inputs = tokenizer(smiles_list, padding=True, truncation=True, max_length=512, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        vectors = model(**inputs).last_hidden_state[:, 0, :].detach().cpu().numpy()
    return vectors.astype(np.float32)

# cuVS expects a CUDA-backed matrix: one row per molecule, one column per embedding dimension.
vectors = cp.asarray(embed_smiles(molecules["smiles"].tolist()), dtype=cp.float32)
molecules

In [ ]:
# exact cosine nearest neighbors for a query molecule
query = pd.DataFrame({"name": ["butanol"], "smiles": ["CCCCO"]})
query_vectors = cp.asarray(embed_smiles(query["smiles"].tolist()), dtype=cp.float32)

index = brute_force.build(vectors, metric="cosine")
distances, neighbors = brute_force.search(index, query_vectors, k=4)

neighbor_ids = cp.asarray(neighbors).get()[0]
hits = molecules.iloc[neighbor_ids].copy()
hits["cosine_similarity"] = 1.0 - cp.asarray(distances).get()[0]
hits

## Joins

cuVS returns neighbor row IDs and distances. Keep metadata in pandas, then join or index back into that table after the GPU search.

In [ ]:
queries = pd.DataFrame({
    "query": ["butanol", "anisole", "salicylic acid"],
    "smiles": ["CCCCO", "COc1ccccc1", "O=C(O)c1ccccc1O"],
})
query_vectors = cp.asarray(embed_smiles(queries["smiles"].tolist()), dtype=cp.float32)
distances, neighbors = brute_force.search(index, query_vectors, k=3)

neighbor_table = pd.DataFrame({
    "query": np.repeat(queries["query"].to_numpy(), 3),
    "rank": np.tile(np.arange(1, 4), len(queries)),
    "neighbor_id": cp.asarray(neighbors).get().ravel(),
    "cosine_similarity": 1.0 - cp.asarray(distances).get().ravel(),
})
molecule_lookup = molecules.reset_index().rename(columns={"index": "neighbor_id"})
neighbor_table.merge(molecule_lookup, on="neighbor_id", how="left")

## CuPy -> NumPy and NumPy -> CuPy

cuVS works with CUDA-backed arrays. CuPy and NumPy convert back and forth easily, which is helpful when metadata, plotting, or CPU-only libraries are part of the workflow.

In [ ]:
host_vectors = cp.asnumpy(vectors)          # GPU -> CPU
back_on_gpu = cp.asarray(host_vectors)      # CPU -> GPU
type(back_on_gpu), type(host_vectors), back_on_gpu.shape

## Ease of using cuVS in embedding workflows

Once your application produces a `float32` embedding matrix, the cuVS part is small: move vectors to the GPU, build an index, and search query vectors. This snippet is the core pattern.

```python
vectors_gpu = cp.asarray(embeddings, dtype=cp.float32)
queries_gpu = cp.asarray(query_embeddings, dtype=cp.float32)

index = brute_force.build(vectors_gpu, metric="cosine")
distances, neighbors = brute_force.search(index, queries_gpu, k=10)
```

## Brute Force vs IVF-Flat

Both are cuVS nearest-neighbor indexes, but they make different tradeoffs.

- **Brute force**: exact, simple, scans every vector. Good as a correctness baseline.
- **IVF-Flat**: approximate, clusters vectors into inverted lists. Use when the library is large and you want faster search with tunable recall.

In [ ]:
build_params = ivf_flat.IndexParams(n_lists=4, metric="cosine", kmeans_trainset_fraction=1.0)
search_params = ivf_flat.SearchParams(n_probes=2)

ivf_index = ivf_flat.build(build_params, vectors)
ivf_distances, ivf_neighbors = ivf_flat.search(search_params, ivf_index, query_vectors, k=3)

exact = cp.asarray(neighbors).get()
approx = cp.asarray(ivf_neighbors).get()
recall = sum(len(set(e) & set(a)) for e, a in zip(exact, approx)) / exact.size

print(f"IVF-Flat recall@3 vs brute force: {recall:.3f}")
pd.DataFrame({
    "query": np.repeat(queries["query"].to_numpy(), 3),
    "rank": np.tile(np.arange(1, 4), len(queries)),
    "neighbor_id": approx.ravel(),
    "cosine_similarity": 1.0 - cp.asarray(ivf_distances).get().ravel(),
}).merge(molecule_lookup, on="neighbor_id", how="left")